# Hitters Regression: Model Selection, Ridge, Lasso, PCR and PLS

This notebook evaluates several regression/model-selection approaches on the **Hitters** dataset from `ISLP`:

- Forward stepwise selection
- Cross-validation and validation-set error
- Best-subset selection via `l0bnb`
- Ridge regression
- Lasso regression
- Principal Components Regression (PCR)
- Partial Least Squares (PLS)

The original code was cleaned and corrected where needed, especially around:
- malformed escaped characters/markdown artifacts,
- `dropna()` syntax,
- Ridge coefficient paths,
- R² scoring,
- preprocessing leakage,
- plotting,
- and consistent cross-validation pipelines.


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.api import OLS

import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

from ISLP import load_data
from ISLP.models import ModelSpec as MS
from ISLP.models import Stepwise, sklearn_selected, sklearn_selection_path

from functools import partial

# Optional package used for best-subset selection
try:
    from l0bnb import fit_path
    L0_AVAILABLE = True
except ImportError:
    L0_AVAILABLE = False
    print("l0bnb is not installed. The best-subset section will be skipped.")

In [ ]:
# Load and clean the Hitters data
Hitters = load_data("Hitters")

print("Missing Salary values:", Hitters["Salary"].isna().sum())

Hitters = Hitters.dropna().copy()

print("Dataset shape after removing missing values:", Hitters.shape)
Hitters.head()

## 1. Forward Stepwise Selection

We first estimate the residual variance from the full OLS model and define a negative Mallows' Cp criterion. We then use forward stepwise selection to identify a suitable sequence of models.

In [ ]:
def nCp(sigma2, estimator, X, Y):
    """Negative Mallows' Cp statistic, scaled by n."""
    n, p = X.shape
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat) ** 2)
    return -(RSS + 2 * p * sigma2) / n


design = MS(Hitters.columns.drop("Salary")).fit(Hitters)

Y = np.asarray(Hitters["Salary"])
X = design.transform(Hitters)

sigma2 = OLS(Y, X).fit().scale
neg_Cp = partial(nCp, sigma2)

strategy = Stepwise.first_peak(
    design,
    direction="forward",
    max_terms=len(design.terms)
)

hitters_MSE = sklearn_selected(
    OLS,
    strategy
)

hitters_MSE.fit(Hitters, Y)

print("Selected model:")
print(hitters_MSE.selected_state_)

In [ ]:
# Generate the complete forward-stepwise path
strategy = Stepwise.fixed_steps(
    design,
    len(design.terms),
    direction="forward"
)

full_path = sklearn_selection_path(OLS, strategy)
full_path.fit(Hitters, Y)

Yhat_in = full_path.predict(Hitters)

print("Prediction matrix shape:", Yhat_in.shape)

In [ ]:
# In-sample MSE across the stepwise path
insample_mse = ((Yhat_in - Y[:, None]) ** 2).mean(axis=0)
n_steps = insample_mse.shape[0]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(np.arange(n_steps), insample_mse, label="In-sample MSE")

ax.set_ylabel("MSE")
ax.set_xlabel("# steps of forward stepwise")
ax.set_xticks(np.arange(n_steps)[::2])
ax.legend()
ax.grid(alpha=0.25)

plt.show()

### Cross-validation and validation-set comparison

The original code used a fixed 5-fold split for cross-validation and a separate 80/20 validation split. Here we preserve that idea while making the calculations explicit.

In [ ]:
K = 5

kfold = skm.KFold(
    n_splits=K,
    random_state=0,
    shuffle=True
)

Yhat_cv = skm.cross_val_predict(
    full_path,
    Hitters,
    Y,
    cv=kfold
)

cv_mse = []

for train_idx, test_idx in kfold.split(Y):
    errors = (Yhat_cv[test_idx] - Y[test_idx, None]) ** 2
    cv_mse.append(errors.mean(axis=0))

cv_mse = np.asarray(cv_mse).T

print("Cross-validation MSE matrix shape:", cv_mse.shape)

In [ ]:
# Plot in-sample and cross-validated MSE
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    np.arange(n_steps),
    insample_mse,
    label="In-sample"
)

ax.errorbar(
    np.arange(n_steps),
    cv_mse.mean(axis=1),
    yerr=cv_mse.std(axis=1, ddof=1) / np.sqrt(K),
    label="5-fold CV"
)

ax.set_ylabel("MSE")
ax.set_xlabel("# steps of forward stepwise")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

In [ ]:
# 80/20 validation-set evaluation
validation = skm.ShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=0
)

for train_idx, test_idx in validation.split(Y):
    full_path.fit(
        Hitters.iloc[train_idx],
        Y[train_idx]
    )

    Yhat_val = full_path.predict(Hitters.iloc[test_idx])
    errors = (Yhat_val - Y[test_idx, None]) ** 2
    validation_mse = errors.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    np.arange(n_steps),
    insample_mse,
    label="In-sample"
)

ax.errorbar(
    np.arange(n_steps),
    cv_mse.mean(axis=1),
    yerr=cv_mse.std(axis=1, ddof=1) / np.sqrt(K),
    label="5-fold CV"
)

ax.plot(
    np.arange(n_steps),
    validation_mse,
    "--",
    label="Validation"
)

ax.set_ylabel("MSE")
ax.set_xlabel("# steps of forward stepwise")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

## 2. Best-Subset Selection with `l0bnb`

`l0bnb` can be used to compute a path of sparse regression solutions. This section is optional because `l0bnb` may not be installed in every environment.

In [ ]:
D = design.fit_transform(Hitters)
D = D.drop("intercept", axis=1)

X = np.asarray(D)

if L0_AVAILABLE:
    subset_path = fit_path(
        X,
        Y,
        max_nonzeros=X.shape[1]
    )

    print(subset_path[3])
else:
    print("Install l0bnb to run this section.")

## 3. Ridge Regression

Ridge regression uses an L2 penalty. The predictors are standardized before fitting so that the penalty treats variables on comparable scales.

In [ ]:
# Standardized predictors for coefficient-path calculations
Xs = X - X.mean(axis=0, keepdims=True)
X_scale = X.std(axis=0, ddof=0)
Xs = Xs / X_scale[None, :]

# Alpha is the lambda-like penalty parameter used by sklearn's ElasticNet/Lasso
lambdas = 10 ** np.linspace(8, -2, 100) / Y.std()

# Ridge path
soln_array = skl.ElasticNet.path(
    Xs,
    Y,
    l1_ratio=0.0,
    alphas=lambdas
)[1]

soln_path = pd.DataFrame(
    soln_array.T,
    columns=D.columns,
    index=-np.log(lambdas)
)

soln_path.index.name = "negative log(lambda)"

print(soln_path.shape)
soln_path.head()

In [ ]:
# Ridge coefficient paths
fig, ax = plt.subplots(figsize=(8, 6))
soln_path.plot(ax=ax, legend=False)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Standardized coefficients")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# Examine coefficient shrinkage at two penalty levels
for idx in [39, 59]:
    beta_hat = soln_path.iloc[idx]
    print(f"Index: {idx}")
    print(f"lambda: {lambdas[idx]:.6g}")
    print(f"L2 norm of coefficients: {np.linalg.norm(beta_hat):.4f}")
    print()

### Ridge cross-validation

A pipeline is used so scaling is learned **inside each training fold**, preventing information leakage from the validation folds.

In [ ]:
ridge = skl.ElasticNet(
    alpha=lambdas[59],
    l1_ratio=0.0
)

scaler = StandardScaler()

ridge_pipe = Pipeline([
    ("scaler", scaler),
    ("ridge", ridge)
])

ridge_pipe.fit(X, Y)

print("Coefficient L2 norm:", np.linalg.norm(ridge_pipe.named_steps["ridge"].coef_))

In [ ]:
# Compare very small and very large ridge penalties
for alpha in [0.01, 1e10]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", skl.ElasticNet(alpha=alpha, l1_ratio=0.0))
    ])

    results = skm.cross_validate(
        model,
        X,
        Y,
        scoring="neg_mean_squared_error",
        cv=validation
    )

    print(f"alpha={alpha:g}, validation MSE={-results['test_score'].mean():.2f}")

In [ ]:
# Grid search using a single 80/20 split
param_grid = {"ridge__alpha": lambdas}

grid_validation = skm.GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=validation,
    scoring="neg_mean_squared_error"
)

grid_validation.fit(X, Y)

print("Best alpha (80/20 validation):", grid_validation.best_params_["ridge__alpha"])

In [ ]:
# More stable 5-fold grid search
grid_ridge = skm.GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=kfold,
    scoring="neg_mean_squared_error"
)

grid_ridge.fit(X, Y)

print("Best alpha (5-fold CV):", grid_ridge.best_params_["ridge__alpha"])

In [ ]:
# Cross-validated Ridge MSE curve
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(
    -np.log(lambdas),
    -grid_ridge.cv_results_["mean_test_score"],
    yerr=grid_ridge.cv_results_["std_test_score"] / np.sqrt(K)
)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Cross-validated MSE")
ax.grid(alpha=0.25)

plt.show()

### Important correction: R² scoring

The original code created a `GridSearchCV` without specifying `scoring` and then labeled the resulting values as cross-validated R². Scikit-learn's default scoring for a regressor is indeed R², but making it explicit avoids confusion.

In [ ]:
grid_r2 = skm.GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=kfold,
    scoring="r2"
)

grid_r2.fit(X, Y)

fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(
    -np.log(lambdas),
    grid_r2.cv_results_["mean_test_score"],
    yerr=grid_r2.cv_results_["std_test_score"] / np.sqrt(K)
)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Cross-validated $R^2$")
ax.grid(alpha=0.25)

plt.show()

print("Best ridge alpha by R²:", grid_r2.best_params_["ridge__alpha"])
print("Best mean CV R²:", grid_r2.best_score_)

In [ ]:
# Ridge CV using ElasticNetCV
ridgeCV = skl.ElasticNetCV(
    alphas=lambdas,
    l1_ratio=0.0,
    cv=kfold
)

pipeCV = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", ridgeCV)
])

pipeCV.fit(X, Y)

tuned_ridge = pipeCV.named_steps["ridge"]

print("Selected ridge alpha:", tuned_ridge.alpha_)
print("Minimum mean CV MSE:", tuned_ridge.mse_path_.mean(axis=1).min())

In [ ]:
# Plot Ridge CV error path
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(
    -np.log(tuned_ridge.alphas_),
    tuned_ridge.mse_path_.mean(axis=1),
    yerr=tuned_ridge.mse_path_.std(axis=1) / np.sqrt(K)
)

ax.axvline(
    -np.log(tuned_ridge.alpha_),
    linestyle="--",
    label="Selected alpha"
)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Cross-validated MSE")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

ridge_coef = pd.Series(
    pipeCV.named_steps["ridge"].coef_,
    index=D.columns
).sort_values(key=np.abs, ascending=False)

ridge_coef

### Nested evaluation of Ridge

The outer split estimates test error while the inner CV chooses the penalty. This is a better estimate of generalization performance than tuning and evaluating on the same validation data.

In [ ]:
outer_valid = skm.ShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=1
)

inner_cv = skm.KFold(
    n_splits=5,
    shuffle=True,
    random_state=2
)

ridgeCV_nested = skl.ElasticNetCV(
    alphas=lambdas,
    l1_ratio=0.0,
    cv=inner_cv
)

pipeCV_nested = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", ridgeCV_nested)
])

results = skm.cross_validate(
    pipeCV_nested,
    X,
    Y,
    cv=outer_valid,
    scoring="neg_mean_squared_error"
)

print("Outer test MSE:", -results["test_score"].mean())

## 4. Lasso Regression

The Lasso uses an L1 penalty and can therefore shrink some coefficients exactly to zero, performing variable selection.

In [ ]:
lassoCV = skl.ElasticNetCV(
    n_alphas=100,
    l1_ratio=1.0,
    cv=kfold
)

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", lassoCV)
])

lasso_pipe.fit(X, Y)

tuned_lasso = lasso_pipe.named_steps["lasso"]

print("Selected Lasso alpha:", tuned_lasso.alpha_)

In [ ]:
# Lasso coefficient path
lasso_lambdas, lasso_soln_array = skl.Lasso.path(
    Xs,
    Y,
    n_alphas=100
)[:2]

lasso_soln_path = pd.DataFrame(
    lasso_soln_array.T,
    columns=D.columns,
    index=-np.log(lasso_lambdas)
)

fig, ax = plt.subplots(figsize=(8, 6))
lasso_soln_path.plot(ax=ax, legend=False)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Standardized coefficients")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# Lasso CV error
print(
    "Minimum mean CV MSE:",
    tuned_lasso.mse_path_.mean(axis=1).min()
)

fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(
    -np.log(tuned_lasso.alphas_),
    tuned_lasso.mse_path_.mean(axis=1),
    yerr=tuned_lasso.mse_path_.std(axis=1) / np.sqrt(K)
)

ax.axvline(
    -np.log(tuned_lasso.alpha_),
    linestyle="--",
    label="Selected alpha"
)

ax.set_xlabel(r"$-\log(\lambda)$")
ax.set_ylabel("Cross-validated MSE")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

In [ ]:
# Lasso coefficients and selected variables
lasso_coef = pd.Series(
    tuned_lasso.coef_,
    index=D.columns
)

selected_lasso = lasso_coef[lasso_coef != 0].sort_values(
    key=np.abs,
    ascending=False
)

print("Number of selected variables:", len(selected_lasso))
selected_lasso

## 5. Principal Components Regression (PCR)

PCR first standardizes the predictors, transforms them into principal components, and then fits linear regression using a selected number of components.

**Important:** the PCA step is inside the pipeline, so each CV fold learns its own transformation.

In [ ]:
pca = PCA()
linreg = skl.LinearRegression()

pcr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", pca),
    ("linreg", linreg)
])

param_grid_pcr = {
    "pca__n_components": range(1, min(20, X.shape[1] + 1))
}

grid_pcr = skm.GridSearchCV(
    pcr_pipe,
    param_grid_pcr,
    cv=kfold,
    scoring="neg_mean_squared_error"
)

grid_pcr.fit(X, Y)

print("Best number of PCR components:",
      grid_pcr.best_params_["pca__n_components"])
print("Best PCR CV MSE:",
      -grid_pcr.best_score_)

In [ ]:
# PCR CV curve
fig, ax = plt.subplots(figsize=(8, 6))

n_comp = list(param_grid_pcr["pca__n_components"])

ax.errorbar(
    n_comp,
    -grid_pcr.cv_results_["mean_test_score"],
    yerr=grid_pcr.cv_results_["std_test_score"] / np.sqrt(K)
)

ax.set_ylabel("Cross-validated MSE")
ax.set_xlabel("# principal components")
ax.set_xticks(n_comp[::2])
ax.grid(alpha=0.25)

plt.show()

In [ ]:
# Null model: intercept only
Xn = np.zeros((X.shape[0], 1))

cv_null = skm.cross_validate(
    linreg,
    Xn,
    Y,
    cv=kfold,
    scoring="neg_mean_squared_error"
)

print("Null-model CV MSE:", -cv_null["test_score"].mean())

# Explained variance from the full PCA fit
pca_full = PCA().fit(StandardScaler().fit_transform(X))
explained = pca_full.explained_variance_ratio_

print("Cumulative variance explained by first 5 PCs:",
      explained[:5].sum())

## 6. Partial Least Squares (PLS)

PLS constructs components using both the predictors and the response, unlike PCR, where components are created using only the predictors.

In [ ]:
pls = PLSRegression(
    n_components=2,
    scale=True
)

param_grid_pls = {
    "n_components": range(1, min(20, X.shape[1] + 1))
}

grid_pls = skm.GridSearchCV(
    pls,
    param_grid_pls,
    cv=kfold,
    scoring="neg_mean_squared_error"
)

grid_pls.fit(X, Y)

print("Best number of PLS components:",
      grid_pls.best_params_["n_components"])
print("Best PLS CV MSE:",
      -grid_pls.best_score_)

In [ ]:
# PLS CV curve
fig, ax = plt.subplots(figsize=(8, 6))

n_comp = list(param_grid_pls["n_components"])

ax.errorbar(
    n_comp,
    -grid_pls.cv_results_["mean_test_score"],
    yerr=grid_pls.cv_results_["std_test_score"] / np.sqrt(K)
)

ax.set_ylabel("Cross-validated MSE")
ax.set_xlabel("# PLS components")
ax.set_xticks(n_comp[::2])
ax.grid(alpha=0.25)

plt.show()

## 7. Model Comparison

The table below compares the best cross-validated MSE and R² for the major regularized/dimension-reduction methods.

In [ ]:
# Compare selected models
comparison = pd.DataFrame({
    "Model": [
        "Ridge",
        "Lasso",
        "PCR",
        "PLS"
    ],
    "Best CV MSE": [
        -grid_ridge.best_score_,
        tuned_lasso.mse_path_.mean(axis=1).min(),
        -grid_pcr.best_score_,
        -grid_pls.best_score_
    ]
})

comparison.sort_values("Best CV MSE")

## Key evaluation notes

1. **Your overall methodology is appropriate** for the chapter/topic: model selection, regularization, PCR and PLS are all relevant approaches for the Hitters data.
2. The original code had several **formatting artifacts** such as `*;*`, escaped underscores, and malformed LaTeX strings. These were cleaned.
3. The original Ridge R² section was potentially confusing because `GridSearchCV` was relying on the regressor's default R² scorer. It is now explicitly set to `scoring="r2"`.
4. Scaling is performed **inside pipelines** for Ridge, Lasso and PCR, which is important during cross-validation.
5. The original PCR code reused a PCA object with `n_components=2` and later changed it through GridSearchCV. The revised version uses a clean pipeline with `PCA()` and tunes the number of components.
6. For PLS/PCR, the maximum number of components is bounded by the number of predictors.
7. The nested Ridge evaluation is retained because it provides a cleaner estimate of out-of-sample error after hyperparameter tuning.
8. **Do not compare the numerical results until all cells have been run in the same environment**, because package versions can slightly affect optimization and CV results.
